## Federated model with per-client SMOTE

This notebook trains the federated model on the SceneFake dataset. Two points a reviewer may look for are handled explicitly:

1. **Class imbalance (SMOTE).** SceneFake is roughly 1 real : 4 fake. We apply **SMOTE per client** - each client oversamples the minority (real) class on its *own* local training data before training. This respects the federated setting (no raw data is pooled) and is applied to training data only, never to the evaluation set.

2. **Features (LFCC vs MFCC).** The pipeline is an LFCC-style cepstral pipeline (frame -> window -> FFT -> filterbank -> log -> DCT). Because Librosa has no built-in LFCC function, the extractor uses `librosa.feature.mfcc` as a proxy; MFCC and LFCC differ *only* in the filterbank spacing (mel vs linear). See the repository `docs/feature_note.md` and `src/features.py` for the drop-in true-LFCC extractor.

In [ ]:
!pip install -q imbalanced-learn
import os, random, numpy as np, torch
import librosa
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_curve, confusion_matrix

# SMOTE is used to correct the class imbalance in SceneFake (~1 real : 4 fake).
# In federated learning we apply it PER CLIENT (see local_train below).
from imblearn.over_sampling import SMOTE


In [ ]:
def get_paths_labels(path, label):
    return [(os.path.join(path, f), label) for f in os.listdir(path) if f.endswith('.wav')]

train_real = get_paths_labels("/kaggle/input/scenefake/train/real", 0)
train_fake = get_paths_labels("/kaggle/input/scenefake/train/fake", 1)
test_real  = get_paths_labels("/kaggle/input/scenefake/eval/real", 0)
test_fake  = get_paths_labels("/kaggle/input/scenefake/eval/fake", 1)

train_data = train_real + train_fake
test_data  = test_real + test_fake

random.shuffle(train_data)
random.shuffle(test_data)

train_paths, y_train = zip(*train_data)
test_paths, y_test = zip(*test_data)


In [ ]:
def extract_lfcc_flat(path, sr=16000, n_mfcc=20, max_len=150):
    y, _ = librosa.load(path, sr=sr)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    feat = np.vstack([mfcc, delta, delta2])
    if feat.shape[1] >= max_len:
        feat = feat[:, :max_len]
    else:
        feat = np.pad(feat, ((0, 0), (0, max_len - feat.shape[1])), mode='constant')
    return feat.flatten()  # shape: (60 * max_len,)


In [ ]:
class DNNSpeaker(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
def partition_clients(paths, labels, num_clients=5):
    data = list(zip(paths, labels))
    random.shuffle(data)
    parts = np.array_split(data, num_clients)
    return [(list(p[:, 0]), list(map(int, p[:, 1]))) for p in parts]

clients = partition_clients(train_paths, y_train, num_clients=5)


In [ ]:
# ============================================================
# Per-client local training WITH SMOTE (federated-correct).
#
# Why SMOTE is here and not on the pooled data:
#   In federated learning each client only ever sees its OWN data. Balancing a
#   pooled dataset would require sharing raw samples, breaking the federated
#   principle. Each client balances its own local partition, trains, and the
#   server then averages the weights (FedAvg).
#
# Why it is safe (no leakage):
#   SMOTE is applied ONLY to client training data. The evaluation set (X_test)
#   is never resampled - see the evaluation cell below.
#
# k_neighbors guard:
#   After the 5-way split a client's minority ('real', label 0) count can be
#   small; we set k = min(5, n_minority - 1) and skip balancing for single-class
#   or too-small clients so SMOTE never crashes.
# ============================================================
def local_train(model, data, input_dim, epochs=1, batch_size=16, lr=0.001):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = model.to(device)
    model.train()

    paths, labels = data
    # The dense model uses the FLATTENED cepstral vector: 60 x 150 = 9000 dims.
    X = np.array([extract_lfcc_flat(p) for p in paths])   # shape: (N, 9000)
    y = np.array(labels)                                  # 0 = real, 1 = fake

    # ---- per-client SMOTE (train only) ----
    # Features are already 2-D (N, 9000), so SMOTE applies directly.
    if len(np.unique(y)) > 1:
        n_min = int(np.min(np.bincount(y)))
        if n_min > 1:
            k = min(5, n_min - 1)
            X, y = SMOTE(random_state=42, k_neighbors=k).fit_resample(X, y)

    X = torch.tensor(X, dtype=torch.float32)
    y = torch.tensor(y, dtype=torch.long)

    loader = DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = loss_fn(out, yb)
            loss.backward()
            optimizer.step()

    return model.cpu().state_dict()  # weights returned to server for FedAvg


In [ ]:
def fed_avg(state_dicts):
    avg = {}
    for k in state_dicts[0]:
        avg[k] = torch.stack([sd[k].float() for sd in state_dicts], 0).mean(0)
    return avg


In [ ]:
input_dim = 60 * 150  # 60 features x 150 frames
global_model = DNNSpeaker(input_dim=input_dim)
ROUNDS = 5

for r in range(ROUNDS):
    print(f"Round {r+1}")
    local_weights = []
    for client_data in clients:
        local_model = DNNSpeaker(input_dim)
        local_model.load_state_dict(global_model.state_dict())
        weights = local_train(local_model, client_data, input_dim, epochs=1)
        local_weights.append(weights)
    global_model.load_state_dict(fed_avg(local_weights))


In [ ]:
X_test = np.array([extract_lfcc_flat(p) for p in test_paths])
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to('cuda')
y_test = np.array(y_test)

global_model.eval()
global_model = global_model.to('cuda')
probs, preds = [], []

with torch.no_grad():
    for i in range(0, len(X_test_tensor), 8):
        xb = X_test_tensor[i:i+8]
        out = global_model(xb)
        p = torch.softmax(out, dim=1)
        probs.extend(p[:, 1].cpu().numpy())
        preds.extend(torch.argmax(p, dim=1).cpu().numpy())

probs = np.array(probs)
preds = np.array(preds)


In [ ]:
fpr, tpr, _ = roc_curve(y_test, probs)
fnr = 1 - tpr
eer = fpr[np.nanargmin(np.abs(fnr - tpr))]

print(f"✅ EER: {eer:.4f}")
print("✅ Confusion Matrix:")
print(confusion_matrix(y_test, preds))
